In [1]:
# Celda P1 - Extraccion de precios diarios CRSP (dsf) para los tickers con evento, 2020-2024
import warnings
import pandas as pd
import wrds

warnings.filterwarnings('ignore', category=pd.errors.ChainedAssignmentError)
warnings.filterwarnings('ignore', category=FutureWarning)

db = wrds.Connection()

padron = pd.read_csv('padron_vigencias_2020_2026_ver03_1.csv',
                     keep_default_na=False, na_values=[''])
eventos = pd.read_csv('ranking_tickers_eventos.csv',
                      keep_default_na=False, na_values=[''])

tickers_evento = set(eventos['ticker'])
vidas = padron[padron['ticker'].isin(tickers_evento) & padron['permno'].notna()]
permnos = sorted(set(vidas['permno'].astype(int)))
print(f'tickers con evento: {len(tickers_evento)} | vidas en padron: {len(vidas)} | '
      f'permnos unicos: {len(permnos)}')

# consulta por bloques de 500 permnos (2020-01-01 a 2024-12-31; CRSP no tiene 2025 aun)
piezas = []
for i in range(0, len(permnos), 500):
    bloque = permnos[i:i + 500]
    lista = ','.join(str(p) for p in bloque)
    q = f"""
        select permno, date, prc, ret, vol, shrout
        from crsp.dsf
        where permno in ({lista})
          and date between '2020-01-01' and '2024-12-31'
    """
    piezas.append(db.raw_sql(q))
    print(f'bloque {i // 500 + 1}/{(len(permnos) - 1) // 500 + 1}: '
          f'{len(piezas[-1]):,} renglones', flush=True)

px = pd.concat(piezas, ignore_index=True)
px = px.assign(
    prc=px['prc'].abs(),          # prc negativo = promedio bid/ask; el valor absoluto es el precio
    permno=px['permno'].astype(int),
)
# NOTA: en el archivo DIARIO (dsf) vol viene en acciones (unidades);
# el factor de 100 aplicaba solo al mensual (msf).
print(f'\ntotal: {len(px):,} renglones | permnos con datos: {px.permno.nunique()}')

# validacion ancla: GME (permno 15473?) via ticker en el padron
permno_gme = int(vidas[vidas['ticker'] == 'GME']['permno'].iloc[0])
gme = px[(px.permno == permno_gme) &
         (px.date >= '2021-01-25') & (px.date <= '2021-01-29')]
print('\nGME semana del squeeze (cierres y volumen):')
print(gme[['date', 'prc', 'ret', 'vol']].to_string(index=False))

px.to_csv('precios_dsf_eventos_2020_2024.csv', index=False)
print('\nexportado: precios_dsf_eventos_2020_2024.csv')

Loading library list...
Done
tickers con evento: 1012 | vidas en padron: 1023 | permnos unicos: 977
bloque 1/2: 529,445 renglones
bloque 2/2: 496,497 renglones

total: 1,025,942 renglones | permnos con datos: 977

GME semana del squeeze (cierres y volumen):
      date        prc       ret          vol
2021-01-25      76.79  0.181203  177874000.0
2021-01-26     147.98  0.927074  178587974.0
2021-01-27  347.51001  1.348358   93396666.0
2021-01-28  193.60001 -0.442894   58815805.0
2021-01-29      325.0  0.678719   50566055.0

exportado: precios_dsf_eventos_2020_2024.csv
